<a href="https://colab.research.google.com/github/ranygo12-pixel/AWS/blob/main/Bedrock/Bedrock03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

AWS CLI 설치 및 설정

In [ ]:
%%bash

if command -v aws &> /dev/null
then
    echo "AWS CLI is already installed. Updating..."
    curl "https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip" -o "awscliv2.zip"
    unzip -o awscliv2.zip
    sudo ./aws/install --update
else
    echo "AWS CLI not found. Installing..."
    curl "https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip" -o "awscliv2.zip"
    unzip -o awscliv2.zip
    sudo ./aws/install
fi

# Clean up the downloaded files
rm awscliv2.zip
rm -rf aws

Boto3 설치

In [ ]:
!pip install boto3

Secrets 설정

In [ ]:
import os
import boto3
from google.colab import userdata

# 1. Colab Secrets에서 자격 증명 불러오기 및 환경 변수 설정
os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION'] = 'ap-northeast-2' # 서울 리전 등 인프라가 구성된 리전 입력

# 2. boto3 클라이언트 생성 (환경 변수를 자동 참조함)
s3_client = boto3.client('s3')

# 3. 연동 테스트: S3 버킷 리스트 출력
try:
    response = s3_client.list_buckets()
    print("✅ AWS 연동 성공! S3 버킷 목록:")
    for bucket in response.get('Buckets', []):
        print(f" - {bucket['Name']}")
except Exception as e:
    print(f"❌ 연결 실패: {e}")

Titan Embedding 호출

In [ ]:
import boto3
import json

bedrock = boto3.client('bedrock-runtime', region_name='ap-northeast-2')

def get_embedding(text):
    """텍스트를 벡터로 변환"""
    response = bedrock.invoke_model(
        modelId='amazon.titan-embed-text-v2:0',
        contentType='application/json',
        body=json.dumps({
            "inputText": text,
            "dimensions": 1024,  # 벡터 차원 (256, 512, 1024 선택 가능)
            "normalize": True
        })
    )

    result = json.loads(response['body'].read())
    return result['embedding']  # 1024차원 벡터 반환

# 테스트
embedding = get_embedding("Python 변수명은 snake_case를 사용합니다")
print(f"벡터 길이: {len(embedding)}")
print(f"첫 10개 값: {embedding[:10]}")

코사인 유사도 측정

In [ ]:
import numpy as np

def cosine_similarity(vec1, vec2):
    """코사인 유사도 계산"""
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    return dot_product / (norm1 * norm2)

# 두 코드 스니펫 비교
code1 = "def calculate_total(items): return sum(items)"
code2 = "def get_sum(prices): return sum(prices)"
code3 = "def make_coffee(): return '☕'"


# 임베딩 생성
emb1 = get_embedding(code1)
emb2 = get_embedding(code2)
emb3 = get_embedding(code3)

# 유사도 비교
print(f"code1 vs code2: {cosine_similarity(emb1, emb2):.4f}")  # 높게 나옴
print(f"code1 vs code3: {cosine_similarity(emb1, emb3):.4f}")  # 낮게 나옴

S3 버킷 생성

In [ ]:
# AWS CLI로 생성 (또는 콘솔에서)
!aws s3 mb s3://codebuddy-kb-docs --region ap-northeast-2

# 폴더 구조 생성
!aws s3api put-object --bucket codebuddy-kb-docs --key style-guides/
!aws s3api put-object --bucket codebuddy-kb-docs --key security/
!aws s3api put-object --bucket codebuddy-kb-docs --key frameworks/

샘플 문서 준비하기

In [ ]:
%%bash
cat << EOF > pep8.txt
PEP 8 - Python Code Style Guide

## Naming Conventions
- 변수명: snake_case 사용 (예: user_name, total_count)
- 함수명: snake_case 사용 (예: calculate_total, get_user_data)
- 클래스명: PascalCase 사용 (예: UserProfile, DataProcessor)
- 상수명: UPPER_SNAKE_CASE 사용 (예: MAX_RETRIES, DEFAULT_TIMEOUT)

## Indentation
- 4 spaces per indentation level (tab 사용 금지)

## Line Length
- 최대 79 characters
- 주석/문서열은 72 characters
EOF

In [ ]:
%%bash
cat << EOF > owasp-top10.txt
OWASP Top 10 - Web Application Security Risks

## A01:2021 – Broken Access Control
- 사용자의 권한이 적절하게 관리되지 않아 비인가된 접근이 허용되는 문제.

## A02:2021 – Cryptographic Failures
- 민감한 데이터의 암호화가 부적절하거나 누락되어 데이터 유출로 이어지는 문제.

## A03:2021 – Injection
- SQL, NoSQL, OS 명령어 등 신뢰할 수 없는 데이터가 쿼리나 명령의 일부로 해석되어 실행되는 문제.

## A04:2021 – Insecure Design
- 보안 통제를 적절하게 구현하지 못하여 설계 단계에서부터 보안 취약점이 발생하는 문제.

## A05:2021 – Security Misconfiguration
- 보안 설정이 미흡하거나 기본 설정이 그대로 사용되어 취약점이 노출되는 문제.

## A06:2021 – Vulnerable and Outdated Components
- 알려진 취약점을 가진 오래된 라이브러리나 컴포넌트를 사용하여 발생하는 문제.

## A07:2021 – Identification and Authentication Failures
- 사용자 인증 또는 세션 관리 기능이 제대로 구현되지 않아 계정 탈취가 가능한 문제.

## A08:2021 – Software and Data Integrity Failures
- 소프트웨어 업데이트, 중요한 데이터, CI/CD 파이프라인 등의 무결성 검증이 부족하여 발생하는 문제.

## A09:2021 – Security Logging and Monitoring Failures
- 보안 이벤트 로깅 및 모니터링이 부적절하여 침해 사고 발생 시 탐지 및 대응이 어려운 문제.

## A10:2021 – Server-Side Request Forgery (SSRF)
- 공격자가 서버 측에서 임의의 URL로 요청을 보내 내부 시스템에 접근하거나 정보를 탈취하는 문제.
EOF

로컬에 파일 저장 후 업로드

In [ ]:
# 문서 업로드
!aws s3 cp pep8.txt s3://codebuddy-kb-docs/style-guides/
!aws s3 cp owasp-top10.txt s3://codebuddy-kb-docs/security/

Knowledge Base 생성

In [ ]:
# Knowledge Base ID 확인 방법
import boto3

bedrock_agent = boto3.client('bedrock-agent', region_name='ap-northeast-2')

# 생성된 KB 목록 확인
response = bedrock_agent.list_knowledge_bases()
for kb in response['knowledgeBaseSummaries']:
    print(f"ID: {kb['knowledgeBaseId']}, 이름: {kb['name']}")

데이터 소스 동기화

In [ ]:
def sync_knowledge_base(knowledge_base_id, data_source_id):
    """Knowledge Base 데이터 소스 동기화"""
    response = bedrock_agent.start_ingestion_job(
        knowledgeBaseId=knowledge_base_id,
        dataSourceId=data_source_id
    )
    print(f"동기화 작업 ID: {response['ingestionJob']['ingestionJobId']}")
    return response

# 동기화 상태 확인
def get_sync_status(knowledge_base_id, data_source_id, job_id):
    response = bedrock_agent.get_ingestion_job(
        knowledgeBaseId=knowledge_base_id,
        dataSourceId=data_source_id,
        ingestionJobId=job_id
    )
    status = response['ingestionJob']['status']
    print(f"상태: {status}")
    return status

Chunk 크기 튜닝

In [ ]:
# Knowledge Base 생성 시 청크 설정
chunking_config = {
    "chunkingStrategy": "FIXED_SIZE",
    "fixedSizeChunkingConfiguration": {
        "maxTokens": 500,  # ← 500, 1000, 2000으로 변경 가능
        "overlapPercentage": 20
    }
}

In [ ]:
!aws s3 cp pep8.txt s3://codebuddy-kb-docs/style-guides/ --metadata '{"language":"python","rule_type":"style"}'

메타데이터 추가

In [ ]:
%%bash
cat << 'EOF' > airbnb-style.txt
Airbnb JavaScript Style Guide

## Variables
- `const` for all references; avoid `var`.
- `let` if you plan on reassigning your variable.

## Objects
- Use trailing commas.

## Arrays
- Use array literals.

## Functions
- Use named function expressions.
EOF

In [ ]:
# 문서 업로드
!aws s3 cp airbnb-style.txt s3://codebuddy-kb-docs/style-guides/

In [ ]:
import boto3

s3 = boto3.client('s3', region_name='ap-northeast-2')

def upload_with_metadata(file_path, bucket, key, metadata):
    """메타데이터와 함께 파일 업로드"""
    with open(file_path, 'rb') as f:
        s3.put_object(
            Bucket=bucket,
            Key=key,
            Body=f.read(),
            Metadata=metadata  # ← 여기에 메타데이터!
        )
    print(f"업로드 완료: {key} with {metadata}")

# Python 문서 업로드
upload_with_metadata(
    'pep8.txt',
    'codebuddy-kb-docs',
    'style-guides/python/pep8.txt',
    {'language': 'python', 'rule_type': 'style'}
)

# JavaScript 문서 업로드
upload_with_metadata(
    'airbnb-style.txt',
    'codebuddy-kb-docs',
    'style-guides/javascript/airbnb.txt',
    {'language': 'javascript', 'rule_type': 'style'}
)

KB 질문 테스트

In [ ]:
import boto3

bedrock_agent = boto3.client('bedrock-agent', region_name='ap-northeast-2')
kb_id = userdata.get('KD_ID')

print(f"Knowledge Base ID: {kb_id} 상태 확인 중...")

try:
    # 1. Knowledge Base 상세 정보 가져오기
    kb_response = bedrock_agent.get_knowledge_base(knowledgeBaseId=kb_id)
    kb_status = kb_response['knowledgeBase']['status']
    kb_name = kb_response['knowledgeBase']['name']
    print(f"\nKnowledge Base '{kb_name}' (ID: {kb_id}) 상태: {kb_status}")

    if kb_status == 'ACTIVE':
        # 2. Knowledge Base에 연결된 데이터 소스 목록 가져오기
        ds_response = bedrock_agent.list_data_sources(knowledgeBaseId=kb_id)
        data_sources = ds_response['dataSourceSummaries']

        if data_sources:
            print("\n--- 연결된 데이터 소스 --- ")
            for ds_summary in data_sources:
                data_source_id = ds_summary['dataSourceId']
                data_source_name = ds_summary['name']
                data_source_status = ds_summary['status']
                print(f"  이름: {data_source_name}, ID: {data_source_id}, 상태: {data_source_status}")

                # 3. 각 데이터 소스의 마지막 ingestion job 상태 확인
                ingestion_jobs_response = bedrock_agent.list_ingestion_jobs(
                    knowledgeBaseId=kb_id,
                    dataSourceId=data_source_id
                )
                ingestion_jobs = ingestion_jobs_response['ingestionJobSummaries']

                if ingestion_jobs:
                    latest_job = ingestion_jobs[0] # 최신 작업 가져오기
                    job_status = latest_job['status']
                    # job_type = latest_job['type'] # 'type' 필드는 summary에 없음, 제거
                    job_start_time = latest_job['startedAt']
                    job_end_time = latest_job.get('endedAt', 'N/A')
                    print(f"    - 최신 Ingestion Job: 상태={job_status}, 시작={job_start_time}, " \
                          "종료={job_end_time}")
                    if latest_job.get('failureReasons'):
                        print(f"      실패 이유: {latest_job['failureReasons']}")
                else:
                    print("    - 이 데이터 소스에 대한 Ingestion Job이 없다.")
        else:
            print("\n이 Knowledge Base에 연결된 데이터 소스가 없다.")

    elif kb_status == 'CREATING':
        print("Knowledge Base가 아직 생성 중입니다. 잠시 후 다시 시도해주세요.")
    elif kb_status == 'DELETING':
        print("Knowledge Base가 삭제 중입니다.")
    elif kb_status == 'FAILED':
        print("Knowledge Base 생성에 실패했습니다. AWS 콘솔에서 자세한 오류를 확인해주세요.")
        # Optional: Add more details if get_knowledge_base provides failure reasons

except bedrock_agent.exceptions.ResourceNotFoundException:
    print(f"오류: Knowledge Base ID '{kb_id}'를 찾을 수 없다." \
          "올바른 ID인지 확인하거나 먼저 KB를 생성해야 한다.")
except Exception as e:
    print(f"오류가 발생했다: {e}")

Knowledge Base 데이터 소스 동기화 시작

In [ ]:
# KB ID와 데이터 소스 ID를 사용하여 동기화 시작
# 현재 셀 상단에서 가져온 값들을 사용합니다.
# kb_id: 'D3FXUNCDZB'
# data_source_id: '7CDPPDIGXZ'

print(f"Knowledge Base '{kb_id}'의 데이터 소스 '{data_source_id}' 동기화 시작...")
sync_response = sync_knowledge_base(kb_id, data_source_id)

ingestion_job_id = sync_response['ingestionJob']['ingestionJobId']

# 동기화 상태 폴링 (실제 사용 시에는 더 정교한 로직 필요)
import time

print("\nIngestion Job 완료 대기 중...")
while True:
    status = get_sync_status(kb_id, data_source_id, ingestion_job_id)
    if status == 'COMPLETE':
        print("Ingestion Job이 성공적으로 완료되었습니다!")
        break
    elif status == 'FAILED':
        print("Ingestion Job이 실패했습니다. AWS 콘솔에서 자세한 오류를 확인하세요.")
        break
    elif status == 'ABORTED':
        print("Ingestion Job이 중단되었습니다.")
        break
    else:
        print(f"현재 상태: {status}. 10초 후 다시 확인합니다.")
        time.sleep(10)

print("\n이제 Knowledge Base가 문서 인덱싱을 완료했습니다. 다시 질문을 시도해 보세요.")

KB 질문 테스트

In [ ]:
import boto3

# Initialize the Bedrock Agent Runtime client for Knowledge Base operations
bedrock_agent_runtime = boto3.client('bedrock-agent-runtime', region_name='ap-northeast-2')

def ask_knowledge_base(kb_id, question):
    """Knowledge Base에 질문하고 답변 받기"""
    response = bedrock_agent_runtime.retrieve_and_generate(
        input={'text': question},
        retrieveAndGenerateConfiguration={
            'type': 'KNOWLEDGE_BASE',
            'knowledgeBaseConfiguration': {
                'knowledgeBaseId': kb_id,
                'modelArn': 'global.anthropic.claude-sonnet-4-6'
            }
        }
    )

    return response['output']['text']

# 실행 테스트
questions = [
    "파이썬 변수명 규칙은 무엇인가요?",
    "SQL Injection을 방지하는 방법은?",
    "React에서 컴포넌트 이름은 어떻게 짓나요?"
]

for q in questions:
    print(f"\n {q}")
    print(f"🤖 {ask_knowledge_base(kb_id, q)}")

검색 결과와 Relevance Score 이해하기

In [ ]:
def retrieve_documents_and_scores(kb_id, question):
    """Knowledge Base에서 관련 문서를 검색하고 점수를 반환"""
    response = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=kb_id,
        retrievalQuery={'text': question}
    )
    return response['retrievalResults']

# 실행
print("파이썬 변수명 규칙에 대한 원본 검색 결과를 확인합니다:\n")
retrieved_results = retrieve_documents_and_scores(kb_id, "파이썬 변수명 규칙")

if retrieved_results:
    print("--- Retrieved Results (Raw) ---")
    for i, result in enumerate(retrieved_results):
        score = result['score']
        content = result['content']['text']
        location = result['location']['s3Location']['uri']
        print(f"\n--- Result {i+1} ---")
        print(f"📍 관련성 점수: {score:.4f}")
        print(f"   출처: {location}")
        print(f"   내용: {content.strip()[:500]}...") # 첫 500자만 출력
else:
    print("검색된 결과가 없습니다.")